In [4]:
import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')
CODE, URL = '/content/code', 'github.com/quocthangtrann/computer_vision_ADAS.git'
env = {**os.environ, 'GIT_TERMINAL_PROMPT': '0'}
if not os.path.exists(CODE):
    subprocess.run(f'git clone https://{URL} {CODE}', shell=True, check=True, env=env)
else:
    subprocess.run(f'git -C {CODE} pull', shell=True, check=True, env=env)
subprocess.run(f'rsync -a --exclude .git {CODE}/repo/ /content/drive/MyDrive/capstone_ai/repo/', shell=True, check=True)
try: import segmentation_models_pytorch
except ImportError: subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'segmentation-models-pytorch'], check=True)
os.chdir('/content/drive/MyDrive/capstone_ai')
print('mã mới nhất đã ở capstone_ai/repo/ | thư mục làm việc:', os.getcwd())


Mounted at /content/drive
mã mới nhất đã ở capstone_ai/repo/ | thư mục làm việc: /content/drive/MyDrive/capstone_ai


In [5]:
import os, subprocess
print('thư mục làm việc:', os.getcwd())
print('tiến trình train đang chạy:', subprocess.run('pgrep -af "[t]rain_v3.py"', shell=True, capture_output=True, text=True).stdout.strip() or 'không')
subprocess.run('pkill -f "[t]rain_v3.py"', shell=True)
print('đã dừng tiến trình thừa (nếu có)')
print('repo/train_v3.py có:', os.path.exists('repo/train_v3.py'), '| runs/ có:', os.path.isdir('runs'))

thư mục làm việc: /content/drive/MyDrive/capstone_ai
tiến trình train đang chạy: không
đã dừng tiến trình thừa (nếu có)
repo/train_v3.py có: True | runs/ có: True


In [6]:
import subprocess, sys, time, os
import torch; assert torch.cuda.is_available(), 'Chưa có GPU: đổi runtime sang GPU'
B9, TAG, SEEDS, EPOCHS = 'shard_000,shard_001,shard_002,shard_003', 'b9s4', range(3), 60    # 2 shard: B9='shard_000,shard_001', TAG='b9s2'
T = time.time()
for s in SEEDS:
    n = f'{TAG}_c4aug_s{s}'
    if os.path.exists(f'runs/{n}/final_metrics.json'): print(n, 'đã xong, bỏ qua'); continue
    print(f'\n>>> {n}  [{(time.time() - T) / 60:.1f} phút]', flush=True)
    pr = subprocess.Popen([sys.executable, '-u', 'repo/train_v3.py', '--name', n, '--seed', str(s), '--b9', B9, '--epochs', str(EPOCHS)],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    with open(f'runs/{n}.log', 'a') as lg:
        for ln in pr.stdout:
            lg.write(ln)
            if not ln.startswith('ep ') or ln.split()[1].split('/')[0] in ('1', '10', '20', '30', '40', '50', '60'): print(ln, end='', flush=True)
    rc = pr.wait(); print('mã thoát:', rc, flush=True)
    if rc != 0: print('LỖI: dừng lại, dán đoạn trên cho tôi'); break
else: print(f'\nXONG sau {(time.time() - T) / 60:.1f} phút')



>>> b9s4_c4aug_s0  [0.0 phút]
train 32376 ảnh (thí điểm 1705 + b9 30671) | test 408 | pixel nhãn 254: 76.8%
ep 1/60 loss 0.6289 IoU [0.898, 0.239, 0.615, 0.257] road 0.7411 48s
ep 10/60 loss 0.2918 IoU [0.953, 0.439, 0.762, 0.571] road 0.8612 34s
ep 20/60 loss 0.2634 IoU [0.957, 0.451, 0.786, 0.612] road 0.8729 34s
ep 30/60 loss 0.2480 IoU [0.961, 0.478, 0.8, 0.625] road 0.8814 34s
ep 40/60 loss 0.2320 IoU [0.955, 0.463, 0.801, 0.616] road 0.8650 34s
ep 50/60 loss 0.2264 IoU [0.957, 0.45, 0.806, 0.628] road 0.8676 34s
ep 60/60 loss 0.2229 IoU [0.957, 0.452, 0.805, 0.628] road 0.8700 34s
xong
mã thoát: 0

>>> b9s4_c4aug_s1  [38.0 phút]
train 32376 ảnh (thí điểm 1705 + b9 30671) | test 408 | pixel nhãn 254: 76.8%
ep 1/60 loss 0.9653 IoU [0.831, 0.19, 0.602, 0.239] road 0.6400 42s
ep 10/60 loss 0.2942 IoU [0.946, 0.418, 0.762, 0.547] road 0.8460 34s
ep 20/60 loss 0.2643 IoU [0.96, 0.465, 0.785, 0.613] road 0.8789 34s
ep 30/60 loss 0.2482 IoU [0.956, 0.463, 0.791, 0.604] road 0.8681 34s
e